# Exercise 01 — Risk and Return: The Historical Record

**MSc Finance · Investments · FHNW**

In this exercise you work with the long-run historical record of **total-return indices**
(1927–2025) for several US asset classes, compiled by A. Damodaran (NYU Stern).
You will:

1. Import the data directly from the course GitHub repository — *no local installation needed*.
2. **Compute annual returns yourself** from the index levels.
3. Visualise the **distribution** of returns.
4. Compute **descriptive statistics** (arithmetic vs. geometric mean, volatility, skewness, kurtosis, Sharpe ratio).
5. **Export** your figures and tables.

Run each cell from top to bottom (`Shift+Enter`). Cells marked **🛠 Your turn** ask you to write code.


## 1 · Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# FHNW colour palette (consistent with lecture dashboards)
FHNW = {
    "navy":   "#1B3A5C",
    "blue":   "#0057A4",
    "green":  "#2E7D32",
    "red":    "#C8102E",
    "orange": "#EA8700",
    "yellow": "#FFD500",
}

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

## 2 · Import the data

The data lives in the course repository. We read the Excel file straight from its raw URL,
so there is nothing to download by hand. We tidy up the column names and use the year as the index.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/KroeTiA/Investments/main/Exercise_01/data/histretSP_investments.xls"
SHEET = "Total Return Index_Damadoran"

raw = pd.read_excel(DATA_URL, sheet_name=SHEET)

# Keep only the data columns (drop the trailing source-note column)
raw = raw.loc[:, ~raw.columns.str.contains("Source", case=False, na=False)]
raw.columns = [c.strip() for c in raw.columns]          # trim stray spaces
idx = raw.set_index("Year")

print(f"Loaded {idx.shape[0]} annual observations, {idx.shape[1]} asset classes")
print("Assets:", list(idx.columns))
idx.head()

### Total-return indices over time

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
for col in idx.columns:
    ax.plot(idx.index, idx[col], label=col, lw=1.6)
ax.set_yscale("log")  # log scale: equal vertical distance = equal % change
ax.set_ylabel("Index level (log scale, base = 100 in 1927)")
ax.set_xlabel("Year")
ax.set_title("Historical Total-Return Indices, 1927–2025")
ax.legend(frameon=False, fontsize=8, ncol=2)
fig.tight_layout()
plt.show()

## 3 · From indices to returns  🛠 *Your turn*

A total-return **index** $I_t$ already reinvests all income (dividends, coupons). The annual
**total return** is the relative change in the index from one year to the next:

$$ R_t = \frac{I_t}{I_{t-1}} - 1 $$

**Task:** Compute a DataFrame `rets` of annual returns from `idx`.

*Hint:* pandas has a one-line method for exactly this relative change. Look up
`DataFrame.pct_change()`. Afterwards, the first row will be `NaN` (there is no prior year) —
drop it with `.dropna()`.

In [ ]:
# 🛠 Your turn: compute annual returns and store them in `rets`
# rets = ...

# --- check your result (uncomment once you have `rets`) ---
# print(rets.shape)      # expect (98, 7)
# rets.head()

<details>
<summary>💡 Click for the solution</summary>

```python
rets = idx.pct_change().dropna()
rets.head()
```
</details>

## 4 · Distribution of returns

Once `rets` exists, this cell plots a histogram for each asset class. All panels share the
**same x-axis range and the same bins**, so the differences in *spread* are directly
comparable: a tight, narrow distribution (e.g. T-Bills) signals low volatility, a wide one
(e.g. Small Cap, Gold) signals high volatility.

In [ ]:
n = idx.shape[1]
ncols = 4
nrows = int(np.ceil(n / ncols))

# Common bin edges across ALL assets -> identical x-axis and bar widths everywhere
lo, hi = rets.min().min(), rets.max().max()
bins = np.linspace(lo, hi, 31)

fig, axes = plt.subplots(nrows, ncols, figsize=(13, 3.0 * nrows), sharex=True)
axes = axes.flatten()
for ax, col in zip(axes, rets.columns):
    ax.hist(rets[col], bins=bins, color=FHNW["blue"], alpha=0.85, edgecolor="white")
    ax.axvline(rets[col].mean(), color=FHNW["red"], ls="--", lw=1.3)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel("Annual return")
for ax in axes[n:]:
    ax.set_visible(False)
fig.suptitle("Distribution of Annual Returns (common scale; dashed line = mean)", y=1.02)
fig.tight_layout()
plt.show()

## 5 · Descriptive statistics: mean and standard deviation

We focus on the two most important descriptive statistics: the **mean** (average return)
and the **standard deviation** (a measure of risk). Other statistics (geometric mean,
skewness, kurtosis) follow in Exercise 02.

### 5.1 · By hand — the S&P 500

Before using built-in functions, let us compute both quantities manually for the first
asset, so the formulas are transparent.

The **arithmetic mean** of $n$ returns $R_1, \dots, R_n$ is

$$ \bar{R} = \frac{1}{n} \sum_{t=1}^{n} R_t $$

The **sample variance** measures the average squared deviation from the mean (dividing by
$n-1$, the sample convention):

$$ s^2 = \frac{1}{n-1} \sum_{t=1}^{n} (R_t - \bar{R})^2 $$

The **standard deviation** is its square root, $s = \sqrt{s^2}$, back in the same units as
the returns.

In [ ]:
# Pick the first asset
asset = rets.columns[0]
x = rets[asset]
n = len(x)

# --- Mean: sum of returns divided by the number of observations
mean_manual = x.sum() / n

# --- Variance: average squared deviation from the mean, using (n - 1)
squared_devs = (x - mean_manual) ** 2
var_manual = squared_devs.sum() / (n - 1)

# --- Standard deviation: square root of the variance
std_manual = var_manual ** 0.5

print(f"Asset: {asset}")
print(f"Observations (n):     {n}")
print(f"Arithmetic mean:      {mean_manual:.4f}  ({mean_manual:.2%})")
print(f"Sample variance:      {var_manual:.4f}")
print(f"Standard deviation:   {std_manual:.4f}  ({std_manual:.2%})")

### 5.2 · The shortcut — built-in functions

pandas provides these computations directly. `.mean()`, `.var()` and `.std()` use the
sample convention ($n-1$) by default, so the results match our manual calculation exactly.

In [ ]:
# Same three numbers, via built-in methods
print(f"Mean:   {x.mean():.4f}")
print(f"Var:    {x.var():.4f}")     # ddof=1 (n-1) by default
print(f"Std:    {x.std():.4f}")     # ddof=1 (n-1) by default

# Confirm they agree with the manual computation
assert np.isclose(x.mean(), mean_manual)
assert np.isclose(x.var(),  var_manual)
assert np.isclose(x.std(),  std_manual)
print("\n✓ Manual and built-in results agree.")

### 5.3 · All assets at once

Applied to the whole DataFrame, the same methods return one value per asset. This lets us
compare mean return and risk across all asset classes.

In [ ]:
stats_df = pd.DataFrame({
    "Mean": rets.mean(),
    "Variance": rets.var(),
    "Std. deviation": rets.std(),
})
stats_df.style.format({
    "Mean": "{:.2%}", "Variance": "{:.4f}", "Std. deviation": "{:.2%}",
})

## 6 · Export your results

Run the cell below to download a ZIP with the statistics table (Excel) and the
distribution figure (PNG). In Firefox/Safari you may need to allow pop-ups.

In [ ]:
# Re-create the distribution figure for a clean export (same common scale)
fig_exp, axes = plt.subplots(nrows, ncols, figsize=(13, 3.0 * nrows), sharex=True)
axes = axes.flatten()
for ax, col in zip(axes, rets.columns):
    ax.hist(rets[col], bins=bins, color=FHNW["blue"], alpha=0.85, edgecolor="white")
    ax.set_title(col, fontsize=10); ax.set_xlabel("Annual return")
for ax in axes[len(rets.columns):]:
    ax.set_visible(False)
fig_exp.tight_layout()
fig_exp.savefig("distribution.png", dpi=300, bbox_inches="tight")

with pd.ExcelWriter("descriptive_statistics.xlsx") as w:
    stats_df.to_excel(w, sheet_name="Descriptive")
    rets.to_excel(w, sheet_name="Returns")

# Bundle the two output files into one ZIP for download
import zipfile
with zipfile.ZipFile("exercise_01_results.zip", "w") as z:
    z.write("distribution.png")
    z.write("descriptive_statistics.xlsx")

try:
    from google.colab import files
    files.download("exercise_01_results.zip")
except ImportError:
    print("Not running in Colab — files saved to working directory.")